# Weather and crime in Los Angeles: findings

Reads the processed panel produced by `crime-weather all`. All logic lives in `src/crime_weather/`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from crime_weather.config import load_settings

s = load_settings()
panel = pd.read_parquet(s.path('processed') / 'daily_panel.parquet')
results = pd.read_csv(s.path('processed') / 'model_results.csv')
FIG = s.path('figures')
panel.head()

## 1. Raw relationship: average daily crime by temperature bin

In [ ]:
raw = panel.groupby(['temp_bin', 'crime_category'], observed=True)['crime_count'].mean().unstack()
ax = raw.plot(kind='bar', figsize=(9, 4), rot=0, title='Average daily crimes by max temperature (unadjusted)')
ax.set_xlabel('Max temperature (°F)'); ax.set_ylabel('Crimes per day')
plt.tight_layout(); plt.savefig(FIG / 'raw_by_temp.png', dpi=150)

## 2. Adjusted effects from the count model

Incidence rate ratios relative to a dry 70–80°F day, controlling for month, weekday, year, and holidays.

In [ ]:
res = results[results['term'] != 'rain_day']
fig, ax = plt.subplots(figsize=(9, 4))
for i, (cat, g) in enumerate(res.groupby('category')):
    x = range(len(g))
    ax.errorbar([v + i * 0.12 for v in x], g['irr'], yerr=[g['irr'] - g['ci_low'], g['ci_high'] - g['irr']], fmt='o', label=cat)
ax.set_xticks(range(len(g)), g['term']); ax.axhline(1, color='gray', lw=0.8)
ax.set_ylabel('Incidence rate ratio'); ax.legend(); ax.set_title('Weather effects on daily crime (adjusted)')
plt.tight_layout(); plt.savefig(FIG / 'weather_effects.png', dpi=150)

## 3. Interpretation

_Write 3–5 sentences: what changed between the raw and adjusted views, which crime types respond most to weather, and what the limitations imply._